In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_PATH = Path("..")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
RAW_PATH = PROJECT_PATH / "data" / "raw"

print("Project path:", PROJECT_PATH.resolve())
print("Processed path:", PROCESSED_PATH.resolve())
print("Raw path:", RAW_PATH.resolve())

Project path: D:\fraud-loss-analysis-step1\fraud-loss-analysis
Processed path: D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\processed
Raw path: D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\raw


In [2]:
# Load cleaned transactions
transactions = pd.read_csv(
    PROCESSED_PATH / "fact_transaction_clean.csv"
)

transactions["txn_ts"] = pd.to_datetime(
    transactions["txn_ts"],
    errors="coerce",
    format="mixed"
)

# Load hidden simulation ground truth
ground_truth = pd.read_csv(
    RAW_PATH / "sim_ground_truth.csv"
)

print("Transactions shape:", transactions.shape)
print("Ground truth shape:", ground_truth.shape)

print("\nTransaction columns:")
print(transactions.columns.tolist())

print("\nGround truth columns:")
print(ground_truth.columns.tolist())

print("\nTransaction date range:")
print("Start:", transactions["txn_ts"].min())
print("End:  ", transactions["txn_ts"].max())

print("\nGround truth typology distribution:")
print(
    ground_truth["typology"]
    .value_counts()
)

Transactions shape: (1498353, 14)
Ground truth shape: (10500, 2)

Transaction columns:
['txn_id', 'card_id', 'customer_id', 'merchant_id', 'device_id', 'txn_ts', 'amount', 'channel', 'category', 'merchant_city', 'merchant_country', 'ip_country', 'auth_result', 'customer_home_city']

Ground truth columns:
['txn_id', 'typology']

Transaction date range:
Start: 2025-01-01 00:00:03
End:   2025-12-31 23:59:35

Ground truth typology distribution:
typology
account_takeover          2135
card_testing              2034
cnp_high_risk_merchant    1826
compromised_terminal      1527
cross_border_spike        1492
impossible_travel         1486
Name: count, dtype: int64


In [4]:
# ---------------------------------------------------------
# Strict final validation period
# September 2025 - December 2025
# ---------------------------------------------------------

VALIDATION_START = pd.Timestamp("2025-09-01")
VALIDATION_END = pd.Timestamp("2026-01-01")

validation = transactions[
    (transactions["txn_ts"] >= VALIDATION_START) &
    (transactions["txn_ts"] < VALIDATION_END)
].copy()

print("Validation transactions:", len(validation))

print("\nValidation period:")
print("Start:", validation["txn_ts"].min())
print("End:  ", validation["txn_ts"].max())

print("\nValidation transaction value:")
print(
    f"₹{validation['amount'].sum():,.2f}"
)

# ---------------------------------------------------------
# Attach hidden ground truth ONLY for final evaluation
# ---------------------------------------------------------

validation = validation.merge(
    ground_truth,
    on="txn_id",
    how="left"
)

# A transaction is hidden-truth fraud when a typology exists.
validation["hidden_fraud"] = (
    validation["typology"].notna()
).astype(int)

print("\nHidden-truth validation results:")
print(
    validation["hidden_fraud"]
    .value_counts()
    .rename(index={
        0: "Not hidden fraud",
        1: "Hidden fraud"
    })
)

print("\nHidden fraud transactions:")
print(
    validation["hidden_fraud"].sum()
)

print("\nMissing typology:")
print(
    validation["typology"].isna().sum()
)

Validation transactions: 500414

Validation period:
Start: 2025-09-01 00:00:47
End:   2025-12-31 23:59:35

Validation transaction value:
₹1,333,617,606.22

Hidden-truth validation results:
hidden_fraud
Not hidden fraud    496890
Hidden fraud          3524
Name: count, dtype: int64

Hidden fraud transactions:
3524

Missing typology:
496890


In [5]:
# ---------------------------------------------------------
# Final rules
# ---------------------------------------------------------

# R001: Primary loss-prevention rule
validation["R001_alert"] = (
    (validation["channel"] == "E_COMMERCE") &
    (validation["category"].isin(["Electronics", "Gift Cards"])) &
    (validation["amount"] >= 20000)
).astype(int)

# R002: Card-testing preventive control
validation["R002_alert"] = (
    (validation["channel"] == "E_COMMERCE") &
    (validation["amount"] <= 50)
).astype(int)

print("R001 alerts:", validation["R001_alert"].sum())
print("R002 alerts:", validation["R002_alert"].sum())

print(
    "\nR001 alert rate:",
    f"{validation['R001_alert'].mean() * 100:.4f}%"
)

print(
    "R002 alert rate:",
    f"{validation['R002_alert'].mean() * 100:.4f}%"
)

R001 alerts: 616
R002 alerts: 781

R001 alert rate: 0.1231%
R002 alert rate: 0.1561%


In [6]:
# ---------------------------------------------------------
# Evaluate a rule against hidden ground truth
# ---------------------------------------------------------

def evaluate_rule(df, rule_column, rule_name):
    tp = ((df[rule_column] == 1) & (df["hidden_fraud"] == 1)).sum()
    fp = ((df[rule_column] == 1) & (df["hidden_fraud"] == 0)).sum()
    fn = ((df[rule_column] == 0) & (df["hidden_fraud"] == 1)).sum()
    tn = ((df[rule_column] == 0) & (df["hidden_fraud"] == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    fraud_value_caught = df.loc[
        (df[rule_column] == 1) & (df["hidden_fraud"] == 1),
        "amount"
    ].sum()

    total_hidden_fraud_value = df.loc[
        df["hidden_fraud"] == 1,
        "amount"
    ].sum()

    fraud_value_coverage = (
        fraud_value_caught / total_hidden_fraud_value
        if total_hidden_fraud_value > 0
        else 0
    )

    return {
        "rule": rule_name,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "precision_pct": precision * 100,
        "recall_pct": recall * 100,
        "fpr_pct": fpr * 100,
        "accuracy_pct": accuracy * 100,
        "f1_pct": f1 * 100,
        "fraud_value_caught": fraud_value_caught,
        "fraud_value_coverage_pct": fraud_value_coverage * 100
    }


r001_metrics = evaluate_rule(
    validation,
    "R001_alert",
    "R001 - High-value E-commerce High-risk Category"
)

r002_metrics = evaluate_rule(
    validation,
    "R002_alert",
    "R002 - E-commerce Card Testing"
)

final_metrics = pd.DataFrame([
    r001_metrics,
    r002_metrics
])

final_metrics

,rule,TP,FP,FN,TN,precision_pct,recall_pct,fpr_pct,accuracy_pct,f1_pct,fraud_value_caught,fraud_value_coverage_pct
0,R001 - High-value E-commerce High-risk Category,432,184,3092,496706,70.1299,12.2588,0.0370,99.3453,20.8696,"18,550,201.9800",58.3538
1,R002 - E-commerce Card Testing,716,65,2808,496825,91.6773,20.3178,0.0131,99.4259,33.2636,"18,493.8500",0.0582


In [7]:
# ---------------------------------------------------------
# Final business impact comparison
# ---------------------------------------------------------

REVIEW_COST_PER_ALERT = 150
CUSTOMER_FRICTION_PER_FP = 100
FRAUD_PREVENTION_RATE = 0.80

business_results = []

for metrics in [r001_metrics, r002_metrics]:

    alerts = metrics["TP"] + metrics["FP"]
    expected_fraud_prevented = (
        metrics["fraud_value_caught"] * FRAUD_PREVENTION_RATE
    )

    review_cost = alerts * REVIEW_COST_PER_ALERT
    customer_friction_cost = (
        metrics["FP"] * CUSTOMER_FRICTION_PER_FP
    )

    expected_net_savings = (
        expected_fraud_prevented
        - review_cost
        - customer_friction_cost
    )

    business_results.append({
        "rule": metrics["rule"],
        "alerts": alerts,
        "TP": metrics["TP"],
        "FP": metrics["FP"],
        "precision_pct": metrics["precision_pct"],
        "recall_pct": metrics["recall_pct"],
        "fraud_value_caught": metrics["fraud_value_caught"],
        "expected_fraud_prevented": expected_fraud_prevented,
        "review_cost": review_cost,
        "customer_friction_cost": customer_friction_cost,
        "expected_net_savings": expected_net_savings
    })

business_impact = pd.DataFrame(business_results)

business_impact

,rule,alerts,TP,FP,precision_pct,recall_pct,fraud_value_caught,expected_fraud_prevented,review_cost,customer_friction_cost,expected_net_savings
0,R001 - High-value E-commerce High-risk Category,616,432,184,70.1299,12.2588,"18,550,201.9800","14,840,161.5840",92400,18400,"14,729,361.5840"
1,R002 - E-commerce Card Testing,781,716,65,91.6773,20.3178,"18,493.8500","14,795.0800",117150,6500,"-108,854.9200"


In [8]:
# ---------------------------------------------------------
# Final hidden-ground-truth performance by fraud typology
# ---------------------------------------------------------

typology_results = (
    validation[validation["hidden_fraud"] == 1]
    .groupby("typology")
    .agg(
        fraud_transactions=("txn_id", "count"),
        fraud_value=("amount", "sum"),
        R001_caught=("R001_alert", "sum"),
        R002_caught=("R002_alert", "sum")
    )
    .reset_index()
)

typology_results["R001_recall_pct"] = (
    typology_results["R001_caught"]
    / typology_results["fraud_transactions"]
    * 100
)

typology_results["R002_recall_pct"] = (
    typology_results["R002_caught"]
    / typology_results["fraud_transactions"]
    * 100
)

typology_results["R001_value_caught"] = (
    validation[
        (validation["hidden_fraud"] == 1) &
        (validation["R001_alert"] == 1)
    ]
    .groupby("typology")["amount"]
    .sum()
    .reindex(typology_results["typology"])
    .fillna(0)
    .values
)

typology_results["R002_value_caught"] = (
    validation[
        (validation["hidden_fraud"] == 1) &
        (validation["R002_alert"] == 1)
    ]
    .groupby("typology")["amount"]
    .sum()
    .reindex(typology_results["typology"])
    .fillna(0)
    .values
)

typology_results["R001_value_coverage_pct"] = (
    typology_results["R001_value_caught"]
    / typology_results["fraud_value"]
    * 100
)

typology_results["R002_value_coverage_pct"] = (
    typology_results["R002_value_caught"]
    / typology_results["fraud_value"]
    * 100
)

typology_results.sort_values(
    "fraud_value",
    ascending=False
)

,typology,fraud_transactions,fraud_value,R001_caught,R002_caught,R001_recall_pct,R002_recall_pct,R001_value_caught,R002_value_caught,R001_value_coverage_pct,R002_value_coverage_pct
0,account_takeover,693,"21,956,097.3400",429,0,61.9048,0.0000,"18,457,704.5500",0.0000,84.0664,0.0000
5,impossible_travel,478,"3,869,297.3500",1,0,0.2092,0.0000,"29,242.5500",0.0000,0.7558,0.0000
3,compromised_terminal,506,"2,860,627.6100",1,0,0.1976,0.0000,"30,320.0700",0.0000,1.0599,0.0000
2,cnp_high_risk_merchant,622,"1,698,068.0100",0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,cross_border_spike,509,"1,386,601.0000",1,0,0.1965,0.0000,"32,934.8100",0.0000,2.3752,0.0000
1,card_testing,716,"18,493.8500",0,716,0.0000,100.0000,0.0000,"18,493.8500",0.0000,100.0000


In [9]:
# ---------------------------------------------------------
# Final validation summary
# ---------------------------------------------------------

final_validation_summary = pd.DataFrame([
    {
        "rule_id": "R001",
        "rule_name": "High-value E-commerce High-risk Category",
        "role": "Primary loss-prevention",
        "alerts": r001_metrics["TP"] + r001_metrics["FP"],
        "TP": r001_metrics["TP"],
        "FP": r001_metrics["FP"],
        "FN": r001_metrics["FN"],
        "TN": r001_metrics["TN"],
        "precision_pct": r001_metrics["precision_pct"],
        "recall_pct": r001_metrics["recall_pct"],
        "fpr_pct": r001_metrics["fpr_pct"],
        "accuracy_pct": r001_metrics["accuracy_pct"],
        "f1_pct": r001_metrics["f1_pct"],
        "fraud_value_caught": r001_metrics["fraud_value_caught"],
        "fraud_value_coverage_pct": r001_metrics["fraud_value_coverage_pct"],
        "expected_fraud_prevented": business_impact.loc[0, "expected_fraud_prevented"],
        "review_cost": business_impact.loc[0, "review_cost"],
        "customer_friction_cost": business_impact.loc[0, "customer_friction_cost"],
        "expected_net_savings": business_impact.loc[0, "expected_net_savings"]
    },
    {
        "rule_id": "R002",
        "rule_name": "E-commerce Card Testing",
        "role": "Secondary preventive control",
        "alerts": r002_metrics["TP"] + r002_metrics["FP"],
        "TP": r002_metrics["TP"],
        "FP": r002_metrics["FP"],
        "FN": r002_metrics["FN"],
        "TN": r002_metrics["TN"],
        "precision_pct": r002_metrics["precision_pct"],
        "recall_pct": r002_metrics["recall_pct"],
        "fpr_pct": r002_metrics["fpr_pct"],
        "accuracy_pct": r002_metrics["accuracy_pct"],
        "f1_pct": r002_metrics["f1_pct"],
        "fraud_value_caught": r002_metrics["fraud_value_caught"],
        "fraud_value_coverage_pct": r002_metrics["fraud_value_coverage_pct"],
        "expected_fraud_prevented": business_impact.loc[1, "expected_fraud_prevented"],
        "review_cost": business_impact.loc[1, "review_cost"],
        "customer_friction_cost": business_impact.loc[1, "customer_friction_cost"],
        "expected_net_savings": business_impact.loc[1, "expected_net_savings"]
    }
])

print("FINAL RULE VALIDATION SUMMARY")
display(final_validation_summary)

FINAL RULE VALIDATION SUMMARY


,rule_id,rule_name,role,alerts,TP,FP,FN,TN,precision_pct,recall_pct,fpr_pct,accuracy_pct,f1_pct,fraud_value_caught,fraud_value_coverage_pct,expected_fraud_prevented,review_cost,customer_friction_cost,expected_net_savings
0,R001,High-value E-commerce High-risk Category,Primary loss-prevention,616,432,184,3092,496706,70.1299,12.2588,0.0370,99.3453,20.8696,"18,550,201.9800",58.3538,"14,840,161.5840",92400,18400,"14,729,361.5840"
1,R002,E-commerce Card Testing,Secondary preventive control,781,716,65,2808,496825,91.6773,20.3178,0.0131,99.4259,33.2636,"18,493.8500",0.0582,"14,795.0800",117150,6500,"-108,854.9200"


In [10]:
# ---------------------------------------------------------
# Save final validation artifacts
# ---------------------------------------------------------

# 1. Final rule-level validation summary
final_validation_summary.to_csv(
    PROCESSED_PATH / "final_rule_validation_summary.csv",
    index=False
)

# 2. Final typology-level performance
typology_results.to_csv(
    PROCESSED_PATH / "final_typology_performance.csv",
    index=False
)

# 3. Business impact comparison
business_impact.to_csv(
    PROCESSED_PATH / "final_rule_business_impact.csv",
    index=False
)

# 4. Validation transactions with final rule flags
validation_output = validation[
    [
        "txn_id",
        "card_id",
        "customer_id",
        "merchant_id",
        "txn_ts",
        "amount",
        "channel",
        "category",
        "merchant_country",
        "ip_country",
        "typology",
        "hidden_fraud",
        "R001_alert",
        "R002_alert"
    ]
].copy()

validation_output.to_csv(
    PROCESSED_PATH / "final_validation_transactions.csv",
    index=False
)

print("Saved final validation artifacts:")
for filename in [
    "final_rule_validation_summary.csv",
    "final_typology_performance.csv",
    "final_rule_business_impact.csv",
    "final_validation_transactions.csv"
]:
    path = PROCESSED_PATH / filename
    print(f"✓ {path}")

Saved final validation artifacts:
✓ ..\data\processed\final_rule_validation_summary.csv
✓ ..\data\processed\final_typology_performance.csv
✓ ..\data\processed\final_rule_business_impact.csv
✓ ..\data\processed\final_validation_transactions.csv
